# 0.8.3 — Blender → Remotion : séquence Riis de 3 secondes

Notebook séparé : le notebook PNG reste inchangé. 90 PNG à 30 fps, jamais un épisode complet. Commencer en Draft / 10 %. Le cache doit être sur Drive (montage explicite) pour survivre à la suppression du runtime. Les images et modèles sont des reconstitutions artistiques. Aucun GPU garanti.


In [ ]:
# 1 — Installer/détecter les outils (exécuter une fois)
import os, sys, json, shutil, subprocess, platform, urllib.request, hashlib, tarfile
from pathlib import Path
assert sys.version_info >= (3, 11), "Utiliser Python 3.11 ou plus récent."
REF = "main" # @param ["main", "feat/blender-video-pipeline-0.8.3"]
PROJECT = Path('/content/Destiny-Explained-Studio')
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch',REF,'--single-branch','https://github.com/Alexanst54/Destiny-Explained-Studio.git',str(PROJECT)],check=True)
else:
    if subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Le checkout contient des modifications : les sauvegarder avant de changer REF.')
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin',f'refs/heads/{REF}:refs/remotes/origin/{REF}'],check=True)
    subprocess.run(['git','-C',str(PROJECT),'switch',REF],check=True)
    subprocess.run(['git','-C',str(PROJECT),'merge','--ff-only',f'origin/{REF}'],check=True)
    if subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()!=subprocess.check_output(['git','-C',str(PROJECT),'rev-parse',f'origin/{REF}'],text=True).strip():
        raise RuntimeError('Commits locaux conservés : utiliser une session propre pour cette REF.')
ROOT = PROJECT/'remotion'
if not (ROOT/'blender/import_scene.py').is_file():
    raise RuntimeError('Cette REF ne contient pas le bridge. Pendant la revue, choisir feat/blender-video-pipeline-0.8.3.')
# Node officiel et version verrouillée ; vérifier le SHA publié avant extraction.
version='22.23.4'
existing=shutil.which('node')
if not existing or subprocess.check_output([existing,'-p','process.versions.node'],text=True).strip()!=version:
    arch={'x86_64':'x64','aarch64':'arm64'}.get(platform.machine())
    if not arch: raise RuntimeError('Architecture Node non prise en charge par ce notebook.')
    name=f'node-v{version}-linux-{arch}.tar.xz';base=f'https://nodejs.org/dist/v{version}/'
    cache=Path('/content/destiny-tools');cache.mkdir(exist_ok=True)
    archive=cache/name
    urllib.request.urlretrieve(base+name,archive)
    manifest=urllib.request.urlopen(base+'SHASUMS256.txt').read().decode()
    expected=next(line.split()[0] for line in manifest.splitlines() if line.split()[-1]==name)
    assert hashlib.sha256(archive.read_bytes()).hexdigest()==expected, 'SHA Node incorrect'
    with tarfile.open(archive) as bundle: bundle.extractall(cache,filter='data')
    os.environ['PATH']=str(cache/f'node-v{version}-linux-{arch}'/'bin')+os.pathsep+os.environ['PATH']
NODE=shutil.which('node')
subprocess.run(['npm','ci','--no-audit','--no-fund'],cwd=ROOT,check=True)
BLENDER=shutil.which('blender')
if not BLENDER:
    if os.geteuid()!=0: raise RuntimeError('Installer Blender via le gestionnaire système, puis relancer cette cellule.')
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','blender'],check=True)
    BLENDER=shutil.which('blender')
if not BLENDER: raise RuntimeError('Blender non installé.')
print(subprocess.check_output([BLENDER,'--version'],text=True).splitlines()[0])
print('Commit:',subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip())


In [ ]:
# 2 — Ressources observées (la sélection effective sera dans chaque rapport Blender)
cpus=len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else os.cpu_count()
print('CPU disponibles:',cpus)
print(next((line for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')),'RAM inconnue'))
if shutil.which('nvidia-smi'):
    gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True,timeout=15)
    print(gpu.stdout.strip() if gpu.returncode==0 else 'GPU non accessible : repli CPU possible.')
else: print('Pas de nvidia-smi : Blender vérifiera ses backends, puis choisira CPU si nécessaire.')


In [ ]:
# 2 — Stockage persistant ; conserver ce chemin pour reprendre
USE_DRIVE = True # @param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE=Path('/content/drive/MyDrive/DestinyStudio/blender-video-083')
else:
    CACHE=Path('/content/destiny-blender-video-083')
    print('ATTENTION : cache temporaire, télécharger avant suppression du runtime.')
CACHE.mkdir(parents=True,exist_ok=True)
print('Cache:',CACHE)


In [ ]:
# 3 — Restaurer ou préparer Kokoro et les timestamps existants
# Génère la narration du projet, PAS une vidéo de 35 secondes.
SAVED=CACHE/'narration';READY=ROOT/'out/hybrid_riis_07-ready.json'
READY.parent.mkdir(exist_ok=True)
if (SAVED/'ready.json').is_file():
    shutil.copy2(SAVED/'ready.json',READY)
    shutil.copytree(SAVED/'assets',ROOT/'public/assets/narration',dirs_exist_ok=True)
    print('Narration/timestamps restaurés du cache persistant.')
else:
    # Même préparation Kokoro que le workflow existant. Aucun Kokoro dans le navigateur.
    subprocess.run([sys.executable,'-m','pip','install','virtualenv'],check=True,timeout=300)
    TTS=Path('/content/destiny-tts')
    if not (TTS/'bin/python').exists():subprocess.run([sys.executable,'-m','virtualenv',str(TTS)],check=True,timeout=120)
    subprocess.run([str(TTS/'bin/pip'),'install','torch==2.6.0','--index-url','https://download.pytorch.org/whl/cpu'],check=True,timeout=1200)
    subprocess.run([str(TTS/'bin/pip'),'install','-r',str(ROOT/'tts/requirements-kokoro.txt')],check=True,timeout=1200)
    subprocess.run(['apt-get','install','-y','ffmpeg','espeak-ng'],check=True,timeout=300)
    env={**os.environ,'HF_HUB_DISABLE_XET':'1'}
    subprocess.run([str(TTS/'bin/python'),'scripts/cache-kokoro.py'],cwd=ROOT,env=env,check=True,timeout=1200)
    subprocess.run([str(TTS/'bin/python'),'scripts/narration.py','episodes/hybrid_riis_07.json'],cwd=ROOT,env=env,check=True,timeout=1200)
    SAVED.mkdir(exist_ok=True)
    shutil.copytree(ROOT/'public/assets/narration',SAVED/'assets',dirs_exist_ok=True)
    shutil.copy2(READY,SAVED/'ready.json')
print('Projet prêt:',READY)


In [ ]:
# 4 — Séquence courte, cache vérifié et reprise automatique
DETAIL = "draft" # @param ["draft", "balanced", "cinematic"]
SCALE = 10 # @param [10, 25, 50, 100] {type:"raw"}
DEVICE = "auto" # @param ["auto", "cpu"]
MAX_NEW_FRAMES = 90 # @param {type:"integer"}
FRAME_TIMEOUT = 180 # @param {type:"integer"}
sys.path.insert(0,str(ROOT/'blender'))
from sequence import run
# 100 % exige les 90 mêmes frames valides à une résolution inférieure.
FOLDER,REPORT=run(READY,CACHE/'frames',start=120,count=90,detail=DETAIL,scale=SCALE,device=DEVICE,timeout=FRAME_TIMEOUT,node=NODE,blender=BLENDER,threads=max(1,min(cpus or 1,4)),max_new=MAX_NEW_FRAMES,public=ROOT/'public')
print('Statut:',REPORT['status'],REPORT['completed'],'/ 90 ; nouveaux',REPORT['newFrames'],'cache',REPORT['cacheHits'])
print('Les configurations effectives CPU/GPU et durées sont dans chaque .benchmark.json.')


In [ ]:
# 5 — Remotion : uniquement 120–209, narration/captions/transitions d’origine
if REPORT['status']!='complete':raise RuntimeError('Séquence incomplète : relancer cellule 4, aucun MP4 partiel.')
subprocess.run([NODE,str(ROOT/'node_modules/@remotion/cli/remotion-cli.js'),'browser','ensure'],cwd=ROOT,check=True,timeout=300)
# Image basse résolution ET vidéo basse résolution avant toute Full HD.
command=[NODE,'--experimental-strip-types','scripts/episode.mjs','render',str(READY),'--frames=120-209',f'--scale={SCALE/100}',f'--blender-sequences={FOLDER/"remotion-sequences.json"}']
subprocess.run(command,cwd=ROOT,check=True,timeout=600)
project=json.loads(READY.read_text());VIDEO=ROOT/project['render']['output'].replace('.mp4','-blender.mp4')
shutil.copy2(VIDEO,FOLDER/'riis-3s.mp4')
probe=subprocess.check_output(['ffprobe','-v','error','-show_streams','-show_format','-of','json',str(VIDEO)],text=True,timeout=30)
(FOLDER/'video-probe.json').write_text(probe)
print('MP4:',VIDEO,'octets:',VIDEO.stat().st_size)


In [ ]:
# 6 — Télécharger la séquence, les rapports et le MP4 si disponible
# Exécutable après interruption d’une cellule de rendu si FOLDER est déjà défini.
from google.colab import files
DOWNLOAD=FOLDER if 'FOLDER' in globals() else CACHE
archive=shutil.make_archive('/content/riis-blender-video-083','zip',DOWNLOAD)
files.download(archive)


## Reprise et limites

Même REF, projet ready, cache, détail, résolution, device et threads : les frames terminées sont vérifiées puis réutilisées. Une frame interrompue est recalculée. Changer du code de rendu ou de contenu crée un autre dossier de cache ; ne pas supprimer l’ancien. Après un runtime neuf, remonter Drive et relancer les cellules dans l’ordre. Une coupure pendant l’export copie locale vers public/ est réparée en relançant la cellule 4. Une archive ZIP peut aussi être extraite manuellement dans le dossier CACHE prévu (uniquement une archive de confiance).

Pour CPU/GPU, garder les mêmes réglages et consulter selected/backend/devices, les durées et RSS par frame. Aucun nouveau test GPU/Colab distant n’est revendiqué. Le rendu démarre Blender à chaque frame : fiable mais coûteux ; persistent scene batching est reporté. Cinématique conserve le volume 0.8.2, Draft l’omet conformément au profil. Le MP4 est un extrait du projet original, avec son timing global ; la phrase peut continuer hors des limites de l’extrait. Ne pas publier cet extrait comme une narration complète.

Ne jamais lancer un rendu 35 secondes ici. Rapport et commandes de reprise : `docs/BLENDER-VIDEO-PROGRESS.md`, `remotion/blender/VIDEO-README.md`.
